# Experiment 100 — Locked One-Shot Test Evaluation

Experiment 99 passed every pre-registered validation criterion. This notebook performs the single authorized test evaluation with all architecture, checkpoints, temporal calibration, seeds, and decision rules frozen.

**Primary endpoint:** CrossFitSafe vs Independent over 4 datasets × 4 horizons × 5 seeds.  
**Secondary descriptive ablations:** NaturalSupport, CalGlobal, CalChannelShrink.  
**Forbidden:** test-conditioned alpha, test oracle, checkpoint selection, retraining, or post-test method switching.

The notebook verifies the frozen calibration scalars against Experiment 99, records checkpoint hashes, writes partial results for recovery, and seals completed test results with a marker so rerunning the notebook only reloads the same evaluation.

In [ ]:
# 0. Frozen protocol and one-shot guard
from pathlib import Path
from types import SimpleNamespace,ModuleType
import os,sys,json,math,random,gc,copy,subprocess,hashlib
if sys.version_info < (3,10): raise RuntimeError('Python 3.10 or newer required. Current: '+sys.version)
print('Python runtime:',sys.version); print('Executable:',sys.executable)
PROJECT_ROOT=Path(os.environ.get('FORECAST_PROJECT_ROOT','/data/code/forecast_jepa'))
EXP_NAME='100_locked_one_shot_test_evaluation'; RESULT_DIR=PROJECT_ROOT/'results'/EXP_NAME; RESULT_DIR.mkdir(parents=True,exist_ok=True)
R99_CKPT_ROOT=PROJECT_ROOT/'checkpoints'/'99_five_seed_four_horizon_safe_support_replication'
R98_CKPT_ROOT=PROJECT_ROOT/'checkpoints'/'98_temporal_calibration_safe_support_control'
FROZEN_CAL_PATH=PROJECT_ROOT/'results'/'99_five_seed_four_horizon_safe_support_replication'/'calibration.csv'
FINAL_MARKER=RESULT_DIR/'FINAL_TEST_COMPLETE.json'; PARTIAL_CSV=RESULT_DIR/'test_by_seed.partial.csv'; PARTIAL_NPZ=RESULT_DIR/'test_sample_mse.partial.npz'
DATASETS=['ETTm1','Weather','ETTh2','Exchange']; HORIZONS=[96,192,336,720]; SEEDS=[9801,9802,9803,9804,9805]
METHODS=['Independent','NaturalSupport','CalGlobal','CrossFitSafe','CalChannelShrink']
LOOKBACK=512; LABEL_LEN=48; CHUNK=24; PATCH_LEN=16; STRIDE=8; D_MODEL=128; N_HEADS=4; D_FF=256; N_LAYERS=3; DROPOUT=.10; MAX_DELTA=.5; GATE_INIT_LOGIT=-2.0
BATCH_SIZE=64; EVAL_BATCH=128; SHRINK_LAMBDA=.02; MAX_GATE_DEVIATION=.35; CAL_FRACTION=.12; MIN_CAL_WINDOWS=256; USE_BF16=True; N_BOOT=5000
HORIZON=HORIZONS[0]; N_SLOTS=HORIZON//CHUNK; DATA={}
if not FROZEN_CAL_PATH.exists(): raise FileNotFoundError('Frozen Experiment 99 calibration manifest not found: '+str(FROZEN_CAL_PATH))
print(EXP_NAME,'test rows=',len(DATASETS)*len(HORIZONS)*len(SEEDS)*len(METHODS)); print('Result directory:',RESULT_DIR)
print('SEALED previous result:',FINAL_MARKER.exists())


In [ ]:
# 1. Locate/import the official Time-Series-Library
def valid_root(p):
    p=Path(p).expanduser()
    return p.resolve() if (p/'models'/'PatchTST.py').is_file() else None
candidates=[Path(os.environ.get('TSLIB_ROOT','/data/Time-Series-Library_v2')),Path('/data/Time-Series-Library_v2'),Path('/data/Time-Series-Library'),Path('/data/code/Time-Series-Library'),PROJECT_ROOT/'Time-Series-Library',PROJECT_ROOT.parent/'Time-Series-Library']
TSLIB_ROOT=next((q for q in map(valid_root,candidates) if q is not None),None)
if TSLIB_ROOT is None:
    for parent in (Path('/data/code'),Path('/data'),PROJECT_ROOT.parent):
        if not parent.is_dir(): continue
        hits=list(parent.glob('*/models/PatchTST.py'))+list(parent.glob('*/*/models/PatchTST.py'))
        if hits: TSLIB_ROOT=hits[0].parent.parent.resolve(); break
if TSLIB_ROOT is None:
    dst=PROJECT_ROOT/'Time-Series-Library'; dst.parent.mkdir(parents=True,exist_ok=True)
    if dst.exists() and not valid_root(dst): raise FileNotFoundError(f'Incomplete checkout at {dst}; set TSLIB_ROOT or rename/remove it.')
    print('Official checkout not found; cloning to',dst)
    subprocess.run(['git','clone','--depth','1','https://github.com/thuml/Time-Series-Library.git',str(dst)],check=True)
    TSLIB_ROOT=valid_root(dst)
if TSLIB_ROOT is None: raise FileNotFoundError('Time-Series-Library unavailable')
sys.path.insert(0,str(TSLIB_ROOT)); print('Time-Series-Library:',TSLIB_ROOT)

# Some TSLib installations import optional Reformer code even though PatchTST never uses it.
def reformer_stub(name):
    m=ModuleType(name)
    class ReformerLayer:
        def __init__(self,*a,**k): raise RuntimeError('Reformer stub invoked')
    m.ReformerLayer=ReformerLayer; sys.modules[name]=m
try:
    from models.PatchTST import Model as OfficialPatchTST
except ModuleNotFoundError as e:
    if e.name not in {'reformer','reformer_pytorch'}: raise
    reformer_stub(e.name); from models.PatchTST import Model as OfficialPatchTST
try:
    from data_provider.data_factory import data_provider
except ModuleNotFoundError as e:
    raise ModuleNotFoundError(f'Missing dependency {e.name!r} while importing the official data provider. Install the repository requirements in the Forecast-JEPA kernel.') from e
print('Official PatchTST:',OfficialPatchTST)


In [ ]:
# 2. Reconstruct calibration tail and official test loader; validation is not constructed
import numpy as np,pandas as pd
import torch,torch.nn as nn,torch.nn.functional as F
from torch.utils.data import DataLoader,Subset
from IPython.display import display
if not torch.cuda.is_available(): raise RuntimeError('CUDA required')
device=torch.device('cuda'); torch.set_float32_matmul_precision('high')
print(torch.__version__,torch.cuda.get_device_name(0),'BF16',torch.cuda.is_bf16_supported())
def seed_all(s): random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
def locate(names):
    for root in [Path('/data/dataset'),Path('/data'),TSLIB_ROOT/'dataset']:
        if not root.exists(): continue
        for name in names:
            hits=list(root.glob('**/'+name))
            if hits: return sorted(hits,key=lambda p:len(str(p)))[0].resolve()
    raise FileNotFoundError('Could not locate '+str(names))
CSV={'ETTm1':locate(['ETTm1.csv']),'Weather':locate(['weather.csv']),'ETTh2':locate(['ETTh2.csv']),'Exchange':locate(['exchange_rate.csv','Exchange.csv','exchange.csv'])}
CHANNELS={n:len([c for c in pd.read_csv(p,nrows=2).columns if c.lower() not in {'date','timestamp'}]) for n,p in CSV.items()}
def data_args(n,batch_size,horizon):
    p=CSV[n]; data_name=n if n in {'ETTm1','ETTh2'} else 'custom'; freq={'ETTm1':'t','Weather':'10min','ETTh2':'h','Exchange':'d'}[n]
    return SimpleNamespace(task_name='long_term_forecast',data=data_name,root_path=str(p.parent)+'/',data_path=p.name,features='M',target='OT',freq=freq,seasonal_patterns='Monthly',seq_len=LOOKBACK,label_len=LABEL_LEN,pred_len=horizon,batch_size=batch_size,num_workers=0,embed='timeF',augmentation_ratio=0)
def build_data(horizon):
    out={}
    for n in DATASETS:
        full,_=data_provider(data_args(n,BATCH_SIZE,horizon),'train'); test_set,_=data_provider(data_args(n,EVAL_BATCH,horizon),'test')
        total=len(full); cal_len=max(MIN_CAL_WINDOWS,int(round(CAL_FRACTION*total))); gap=LOOKBACK+horizon; fit_end=total-cal_len-gap; cal_start=fit_end+gap
        cal=Subset(full,range(cal_start,total)); cal_loader=DataLoader(cal,batch_size=EVAL_BATCH,shuffle=False,num_workers=0,drop_last=False,pin_memory=True)
        test_loader=DataLoader(test_set,batch_size=EVAL_BATCH,shuffle=False,num_workers=0,drop_last=False,pin_memory=True)
        out[n]={'cal_loader':cal_loader,'test_loader':test_loader,'channels':CHANNELS[n],'cal_n':len(cal),'test_n':len(test_set)}
        print('H',horizon,n,'cal/test=',len(cal),len(test_set))
    return out


In [ ]:
# 3. Official model exposure and three controlled residual gates
def model_config(C):
    return SimpleNamespace(task_name='long_term_forecast',seq_len=LOOKBACK,pred_len=HORIZON,output_attention=False,patch_len=PATCH_LEN,stride=STRIDE,padding_patch='end',d_model=D_MODEL,n_heads=N_HEADS,e_layers=N_LAYERS,d_ff=D_FF,norm='BatchNorm',activation='gelu',dropout=DROPOUT,fc_dropout=DROPOUT,head_dropout=0.0,individual=False,enc_in=C,factor=1)
class OfficialIndependent(nn.Module):
    def __init__(self,C): super().__init__(); self.C=C; self.H=HORIZON; self.official=OfficialPatchTST(model_config(C))
    def direct(self,x):
        out=self.official(x,None,None,None); out=out[0] if isinstance(out,(tuple,list)) else out; return out
    def expose(self,x):
        means=x.mean(1,keepdim=True).detach(); xn=x-means; stdev=torch.sqrt(torch.var(xn,dim=1,keepdim=True,unbiased=False)+1e-5); xn=xn/stdev
        enc,nvars=self.official.patch_embedding(xn.permute(0,2,1)); enc,_=self.official.encoder(enc)
        enc=enc.reshape(-1,nvars,enc.shape[-2],enc.shape[-1]); head_in=enc.permute(0,1,3,2); yn=self.official.head(head_in); out=yn.permute(0,2,1)
        out=out*stdev[:,0,:].unsqueeze(1)+means[:,0,:].unsqueeze(1)
        return out,head_in.permute(0,1,3,2),stdev.permute(0,2,1)
    def forward(self,x): return self.direct(x)
def controller_features(x):
    z=x.permute(0,2,1); d=z[...,1:]-z[...,:-1]
    return torch.stack([z[...,-1],z.mean(-1),z.std(-1,unbiased=False),z[...,-1]-z[...,0],d.mean(-1),d.std(-1,unbiased=False),d.abs().mean(-1)],-1)
class SampleController(nn.Module):
    def __init__(self): super().__init__(); self.net=nn.Sequential(nn.Linear(7,32),nn.GELU(),nn.Linear(32,N_SLOTS)); nn.init.zeros_(self.net[-1].weight); nn.init.constant_(self.net[-1].bias,GATE_INIT_LOGIT)
    def forward(self,x): return torch.sigmoid(self.net(controller_features(x)))
class ShrinkController(nn.Module):
    def __init__(self,C):
        super().__init__(); self.mean_logit=nn.Parameter(torch.full((1,C,N_SLOTS),GATE_INIT_LOGIT))
        self.deviation=nn.Sequential(nn.Linear(7,32),nn.GELU(),nn.Linear(32,N_SLOTS))
        self.reliability=nn.Sequential(nn.Linear(7,16),nn.GELU(),nn.Linear(16,N_SLOTS))
        nn.init.zeros_(self.deviation[-1].weight); nn.init.zeros_(self.deviation[-1].bias)
        nn.init.zeros_(self.reliability[-1].weight); nn.init.constant_(self.reliability[-1].bias,-1.0)
    def components(self,x):
        f=controller_features(x); mean=torch.sigmoid(self.mean_logit).expand(len(x),-1,-1)
        deviation=MAX_GATE_DEVIATION*torch.tanh(self.deviation(f)); rho=torch.sigmoid(self.reliability(f))
        gate=(mean+rho*deviation).clamp(0.,1.); return gate,mean,deviation,rho
    def forward(self,x): return self.components(x)[0]
class SupportAdapter(nn.Module):
    def __init__(self,C,npatch,method):
        super().__init__(); self.method=method; self.attn=nn.MultiheadAttention(D_MODEL,N_HEADS,dropout=DROPOUT,batch_first=True); self.n1=nn.LayerNorm(D_MODEL); self.ff=nn.Sequential(nn.Linear(D_MODEL,D_FF),nn.GELU(),nn.Dropout(DROPOUT),nn.Linear(D_FF,D_MODEL)); self.n2=nn.LayerNorm(D_MODEL); self.head=nn.Sequential(nn.Flatten(-2),nn.Dropout(DROPOUT),nn.Linear(npatch*D_MODEL,HORIZON)); nn.init.zeros_(self.head[-1].weight); nn.init.zeros_(self.head[-1].bias)
        self.sample=SampleController(); self.mean_logit=nn.Parameter(torch.full((1,C,N_SLOTS),GATE_INIT_LOGIT)); self.shrink=ShrinkController(C)
    def delta(self,z):
        B,C,P,D=z.shape; q=z.permute(0,2,1,3).reshape(B*P,C,D); a,_=self.attn(q,q,q,need_weights=False); q=self.n1(q+a); q=self.n2(q+self.ff(q)); zc=q.reshape(B,P,C,D).permute(0,2,1,3); return MAX_DELTA*torch.tanh(self.head(zc-z))
    def gate(self,x):
        B,_,C=x.shape
        if self.method=='OfficialBoundedFixed': return torch.full((B,C,N_SLOTS),.5,device=x.device)
        if self.method=='OfficialMeanGate': return torch.sigmoid(self.mean_logit).expand(B,-1,-1)
        if self.method=='OfficialShrinkAdaptive': return self.shrink(x)
        return self.sample(x)
class BoundedModel(nn.Module):
    def __init__(self,base,method,npatch):
        super().__init__(); self.base=base; self.method=method; self.adapter=SupportAdapter(base.C,npatch,method)
        for p in self.base.parameters(): p.requires_grad=False
        if method!='OfficialMeanGate': self.adapter.mean_logit.requires_grad=False
        if method!='OfficialSampleAdaptive':
            for p in self.adapter.sample.parameters(): p.requires_grad=False
        if method!='OfficialShrinkAdaptive':
            for p in self.adapter.shrink.parameters(): p.requires_grad=False
    def train(self,mode=True): super().train(mode); self.base.eval(); return self
    def forward(self,x,gate_override=None,diagnostic=False):
        self.base.eval()
        with torch.no_grad(): base,z,sd=self.base.expose(x)
        dn=self.adapter.delta(z.detach()); g=self.adapter.gate(x) if gate_override is None else gate_override; gh=g.unsqueeze(-1).expand(-1,-1,-1,CHUNK).reshape(len(x),self.base.C,HORIZON); corr=gh*dn; pred=base+(corr*sd).permute(0,2,1)
        return (pred,base,g,dn,corr) if diagnostic else pred


In [ ]:
# 4. Audit frozen checkpoints before any metric is computed
def checkpoint_path(kind,h,n,seed):
    rel=Path(f'h{h}')/kind/n/f'{seed}.pth'
    hits=[q/rel for q in [R99_CKPT_ROOT,R98_CKPT_ROOT] if (q/rel).exists()]
    if not hits: raise FileNotFoundError(f'Missing frozen {kind} checkpoint: H={h}, {n}, seed={seed}')
    return hits[0]
def sha256_file(p):
    z=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''): z.update(b)
    return z.hexdigest()
AUDIT=[]
for h in HORIZONS:
 for n in DATASETS:
  for seed in SEEDS:
   bp=checkpoint_path('base',h,n,seed); sp=checkpoint_path('support',h,n,seed)
   AUDIT.append({'horizon':h,'dataset':n,'seed':seed,'base_path':str(bp),'base_sha256':sha256_file(bp),'support_path':str(sp),'support_sha256':sha256_file(sp)})
audit_df=pd.DataFrame(AUDIT); assert len(audit_df)==80 and audit_df.base_sha256.str.len().eq(64).all(); audit_df.to_csv(RESULT_DIR/'frozen_checkpoint_audit.csv',index=False)
frozen_cal=pd.read_csv(FROZEN_CAL_PATH); assert len(frozen_cal)==80
print('Frozen checkpoint pairs:',len(audit_df)); print('Frozen calibration rows:',len(frozen_cal)); display(audit_df.head())


In [ ]:
# 5. Frozen model loading and calibration reconstruction
def unpack(batch):
    x,y=batch[0].float().to(device,non_blocking=True),batch[1].float().to(device,non_blocking=True); return x,y[:,-HORIZON:,:]
def load_frozen(n,seed):
    base=OfficialIndependent(CHANNELS[n]).to(device); bst=torch.load(str(checkpoint_path('base',HORIZON,n,seed)),map_location='cpu',weights_only=False); base.load_state_dict(bst['model']); base.eval()
    x=next(iter(DATA[n]['cal_loader']))[0][:2].float().to(device)
    with torch.no_grad(): _,z,_=base.expose(x)
    m=BoundedModel(base,'OfficialShrinkAdaptive',z.shape[2]).to(device); sst=torch.load(str(checkpoint_path('support',HORIZON,n,seed)),map_location='cpu',weights_only=False); m.adapter.load_state_dict(sst['adapter']); m.eval(); return m
@torch.no_grad()
def collect_components(n,m,split):
    bases=[]; deltas=[]; ys=[]
    for batch in DATA[n][split+'_loader']:
        x,y=unpack(batch)
        with torch.amp.autocast('cuda',dtype=torch.bfloat16,enabled=USE_BF16): pred,base,_,_,_=m(x,diagnostic=True)
        bases.append(base.float().cpu()); deltas.append((pred-base).float().cpu()); ys.append(y.float().cpu())
    return torch.cat(bases),torch.cat(deltas),torch.cat(ys)
def analytic_alpha(base,delta,y,dims):
    e=base-y; return (-(e*delta).sum(dim=dims)/delta.square().sum(dim=dims).clamp_min(1e-12)).clamp(0.,1.)
def mse_alpha(base,delta,y,a): return float((base+delta*a-y).square().mean())
def frozen_policy(base,delta,y):
    ag=analytic_alpha(base,delta,y,(0,1,2)); cut=len(base)//2; accepted=[]
    for fi,ci in [(slice(0,cut),slice(cut,None)),(slice(cut,None),slice(0,cut))]:
        a=analytic_alpha(base[fi],delta[fi],y[fi],(0,1,2))
        if mse_alpha(base[ci],delta[ci],y[ci],a)<mse_alpha(base[ci],delta[ci],y[ci],0.): accepted.append(float(a))
    safe=torch.tensor(float(np.mean(accepted)) if accepted else 0.)
    rawc=analytic_alpha(base,delta,y,(0,1)); energy=delta.square().sum((0,1)); prior=torch.median(energy).clamp_min(1e-12); w=energy/(energy+prior); channel=(w*rawc+(1-w)*ag).view(1,1,-1)
    return {'Independent':torch.tensor(0.),'NaturalSupport':torch.tensor(1.),'CalGlobal':ag,'CrossFitSafe':safe,'CalChannelShrink':channel}


In [ ]:
# 6. ONE-SHOT TEST — completed evaluations are loaded, never recomputed
def key(h,n,m,s): return f'h{h}__{n}__{m}__s{s}'
ROWS=[]; DETAIL={}; ALPHAS=[]
load_csv=(RESULT_DIR/'test_by_seed.csv') if FINAL_MARKER.exists() and (RESULT_DIR/'test_by_seed.csv').exists() else PARTIAL_CSV
load_npz=(RESULT_DIR/'test_sample_mse.npz') if FINAL_MARKER.exists() and (RESULT_DIR/'test_sample_mse.npz').exists() else PARTIAL_NPZ
if load_csv.exists(): ROWS=pd.read_csv(load_csv).to_dict('records')
if load_npz.exists():
    z=np.load(load_npz); DETAIL={k:z[k] for k in z.files}
done={(int(r['horizon']),r['dataset'],r['method'],int(r['seed'])) for r in ROWS if key(int(r['horizon']),r['dataset'],r['method'],int(r['seed'])) in DETAIL}
if FINAL_MARKER.exists(): print('SEALED: loading the previously completed one-shot test; no test forward pass will run.')
else:
 for _h in HORIZONS:
    HORIZON=_h; N_SLOTS=HORIZON//CHUNK; DATA=build_data(HORIZON)
    for n in DATASETS:
      for seed in SEEDS:
        required={(HORIZON,n,m,seed) for m in METHODS}
        if required.issubset(done): continue
        model=load_frozen(n,seed); cb,cd,cy=collect_components(n,model,'cal'); policies=frozen_policy(cb,cd,cy)
        manifest=frozen_cal.query('horizon==@HORIZON and dataset==@n and seed==@seed').iloc[0]
        if abs(float(policies['CalGlobal'])-float(manifest.global_alpha))>1e-6 or abs(float(policies['CrossFitSafe'])-float(manifest.safe_alpha))>1e-6: raise RuntimeError(f'Frozen calibration mismatch: H={HORIZON}, {n}, seed={seed}')
        tb,td,ty=collect_components(n,model,'test')
        for method,a in policies.items():
            if (HORIZON,n,method,seed) in done: continue
            pred=tb+td*a; e=pred-ty; sm=e.square().mean((1,2)).numpy(); ROWS.append({'horizon':HORIZON,'dataset':n,'seed':seed,'method':method,'mse':float(e.square().mean()),'mae':float(e.abs().mean()),'alpha_mean':float(torch.as_tensor(a).mean())}); DETAIL[key(HORIZON,n,method,seed)]=sm
        ALPHAS.append({'horizon':HORIZON,'dataset':n,'seed':seed,'global_alpha':float(policies['CalGlobal']),'safe_alpha':float(policies['CrossFitSafe']),'channel_alpha_mean':float(policies['CalChannelShrink'].mean())})
        pd.DataFrame(ROWS).to_csv(PARTIAL_CSV,index=False); np.savez_compressed(PARTIAL_NPZ,**DETAIL); pd.DataFrame(ALPHAS).to_csv(RESULT_DIR/'frozen_alpha_audit.partial.csv',index=False)
        del model,cb,cd,cy,tb,td,ty; gc.collect(); torch.cuda.empty_cache()
    DATA={}; gc.collect(); torch.cuda.empty_cache()
result_df=pd.DataFrame(ROWS).sort_values(['horizon','dataset','method','seed']).reset_index(drop=True)
expected=len(HORIZONS)*len(DATASETS)*len(SEEDS)*len(METHODS); assert len(result_df)==expected and len(DETAIL)==expected,(len(result_df),len(DETAIL),expected)
result_df.to_csv(RESULT_DIR/'test_by_seed.csv',index=False); np.savez_compressed(RESULT_DIR/'test_sample_mse.npz',**DETAIL)
summary=result_df.groupby(['horizon','dataset','method']).agg(mse=('mse','mean'),std=('mse','std'),mae=('mae','mean'),alpha=('alpha_mean','mean'),seeds=('seed','nunique')).reset_index(); summary.to_csv(RESULT_DIR/'test_summary.csv',index=False); display(summary.sort_values(['horizon','dataset','mse']))


In [ ]:
# 7. Primary locked analysis, sealing, and final dashboard
def mb(a,b,seed,block):
    d=np.asarray(b)-np.asarray(a); rng=np.random.default_rng(seed); vals=[]; n=len(d); block=min(block,n)
    for _ in range(N_BOOT):
        starts=rng.integers(0,max(1,n-block+1),size=int(np.ceil(n/block))); vals.append(np.concatenate([d[s:s+block] for s in starts])[:n].mean())
    return float(d.mean()),float(np.quantile(vals,.025)),float(np.quantile(vals,.975))
B=[]
for h in HORIZONS:
 for n in DATASETS:
  for seed in SEEDS:
   for cand in ['CrossFitSafe','NaturalSupport','CalGlobal','CalChannelShrink']:
    d,lo,hi=mb(DETAIL[key(h,n,'Independent',seed)],DETAIL[key(h,n,cand,seed)],seed+h,h); B.append({'horizon':h,'dataset':n,'comparison':cand+' - Independent','seed':seed,'delta':d,'ci_low':lo,'ci_high':hi,'primary':cand=='CrossFitSafe'})
boot=pd.DataFrame(B); boot.to_csv(RESULT_DIR/'test_paired_bootstrap_by_seed.csv',index=False)
wide=result_df.pivot(index=['horizon','dataset','seed'],columns='method',values='mse').reset_index()
for m in METHODS[1:]: wide[m+'_gain']=(wide.Independent-wide[m])/wide.Independent
def hierarchical_ci(col,horizons,seed):
    rng=np.random.default_rng(seed); strata=[(h,n) for h in horizons for n in DATASETS]; vals=[]
    for _ in range(20000):
        chosen=[strata[i] for i in rng.integers(0,len(strata),len(strata))]; per=[]
        for h,n in chosen:
            a=wide.query('horizon==@h and dataset==@n')[col].to_numpy(); per.append(float(rng.choice(a,size=len(a),replace=True).mean()))
        vals.append(float(np.mean(per)))
    obs=float(wide.query('horizon in @horizons').groupby(['horizon','dataset'])[col].mean().mean()); return {'scope':'all' if len(horizons)>1 else f'H{horizons[0]}','metric':col,'mean':obs,'ci_low':float(np.quantile(vals,.025)),'ci_high':float(np.quantile(vals,.975))}
hier=pd.DataFrame([hierarchical_ci(m+'_gain',hs,1001000+i*10+j) for i,m in enumerate(METHODS[1:]) for j,hs in enumerate([HORIZONS]+[[h] for h in HORIZONS])]); hier.to_csv(RESULT_DIR/'test_hierarchical_bootstrap.csv',index=False)
agg=summary.pivot(index=['horizon','dataset'],columns='method',values='mse').reset_index(); agg['safe_gain']=(agg.Independent-agg.CrossFitSafe)/agg.Independent; agg.to_csv(RESULT_DIR/'test_comparison.csv',index=False)
primary=hier.query("scope=='all' and metric=='CrossFitSafe_gain'").iloc[0]; improved=int((agg.safe_gain>0).sum()); noninferior=int((agg.safe_gain>=-.005).sum()); worst_seed=float((-wide.CrossFitSafe_gain).max())
confirmed=bool(primary.ci_low>0 and improved>=12 and noninferior==16 and worst_seed<=.01)
decision={'experiment':EXP_NAME,'phase':'FINAL_TEST_CLOSED','test_evaluated':True,'primary_endpoint':'CrossFitSafe vs Independent','completed_rows':len(result_df),'expected_rows':400,'improved_cells':improved,'noninferior_cells_at_0.5pct':noninferior,'overall_primary_gain':float(primary['mean']),'overall_primary_95ci':[float(primary.ci_low),float(primary.ci_high)],'worst_seed_level_regression':worst_seed,'test_confirmation_passed':confirmed,'post_test_tuning_permitted':False,'next':'report_results_without_retuning'}
(RESULT_DIR/'test_decision.json').write_text(json.dumps(decision,indent=2)); FINAL_MARKER.write_text(json.dumps({'sealed':True,'experiment':EXP_NAME,'rows':len(result_df),'checkpoint_pairs':len(audit_df),'primary_endpoint':'CrossFitSafe vs Independent','post_test_tuning_permitted':False},indent=2))
print('='*100); print('EXPERIMENT 100 — SEALED ONE-SHOT TEST RESULT'); print('='*100); print('Test evaluated: YES'); print('Post-test tuning permitted: NO'); print('\n[Test aggregate]'); display(summary.sort_values(['horizon','dataset','mse'])); print('\n[Primary cell comparison]'); display(agg); print('\n[Hierarchical bootstrap]'); display(hier); print('\n[Final locked decision]'); print(json.dumps(decision,indent=2))
